# Fine-tune Uzbek embeddings on Colab

Thin orchestrator: all logic lives in the repo's `src/` modules. This trains and
evaluates two models on `sukhrobnurali/uzbek-embedding-pairs`:

- **`uzbek-minilm`** — fine-tunes `paraphrase-multilingual-MiniLM-L12-v2` (weak at Uzbek): large, honest delta.
- **`uzbek-e5-small`** — fine-tunes `intfloat/multilingual-e5-small` (already strong): the flagship, shipped only if it beats base.

**Set the runtime to an A100 GPU** (Runtime → Change runtime type → A100; needs Colab Pro). End-to-end is ~15 min. A free T4 also runs but is slow and needs `config.BATCH_SIZE` lowered (~48) to avoid OOM on the 356k-pair set.

## 1. Get the code and install dependencies

In [ ]:
# Replace with your repo URL if different
!git clone https://github.com/sukhrobnurali/uz-sentance-embedding.git
%cd uz-sentance-embedding
!pip install -q -r requirements.txt

## 2. Authenticate

Needed to push the fine-tuned models and to read the gated FLORES+ eval set.
First accept the FLORES+ terms once: https://huggingface.co/datasets/openlanguagedata/flores_plus
(click *Agree and access repository*). Then run the cell below with a **write** token
(huggingface.co/settings/tokens) and check that the printed username is yours -- this
sets `HF_TOKEN` for the training/eval subprocesses and overrides any stale cached token.

In [ ]:
import os, getpass
from huggingface_hub import logout, whoami

# Clear any stale/other cached token so it can't shadow yours, then set HF_TOKEN
# explicitly -- the `!python` train/eval subprocesses inherit it deterministically.
try:
    logout()
except Exception:
    pass

os.environ["HF_TOKEN"] = getpass.getpass("Paste your HF WRITE token: ")
print("Authenticated as:", whoami(token=os.environ["HF_TOKEN"])["name"])  # must be YOUR username

## 3. MiniLM — the large-delta model

In [ ]:
!python -m src.train --group minilm && python -m src.evaluate --group minilm --stage finetuned

## 4. e5-small — the strong-base flagship

In [ ]:
!python -m src.train --group e5_small && python -m src.evaluate --group e5_small --stage finetuned

## 5. Results — baseline vs fine-tuned, with deltas

In [ ]:
import json
with open('results/metrics.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))